# De datos a decisiones de negocio en un e-commerce multi-país

**Introducción**


El objetivo de este proyecto es evaluar el desempeño de un **E-commerce** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **catalog.csv** → costos de productos, categorías y proveedores  
- **marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python)

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

> **Nota:** proyecto académico con **datos simulados** proporcionados por el curso de Data analyst de Tripleten.
>
> **Reproducibilidad:** los datasets pertenecen al curso y no se redistribuyen; las celdas de carga indican el nombre de cada archivo. Las consultas SQL (Pasos 3 y 4) usan una base de **acceso restringido** del curso; por eso las credenciales se omitieron y los resultados se conservan como salidas guardadas.

## Hallazgos clave

- **Calidad de datos:** se eliminaron 194 de 25,100 filas de pedidos (0.77 %) aplicando 6 reglas documentadas en una bitácora de limpieza.
- **Rentabilidad:** revenue de 9.61 M con margen bruto de 60.2 %, utilidad después de marketing de 30.3 % y ROAS global de 3.35 (punto de equilibrio ≈ 1.66).
- **Producto:** 4 de 7 productos superan 90 % de margen, pero **Laptop-Gaming-16GB tiene margen negativo (−8.6 %)** y Blender-XL-Red y Jacket-Winter-M rinden por debajo del 32 %.
- **Funnel:** conversión final de **3.31 %** (258 de 7,796 usuarios); solo 9.7 % de quienes inician el checkout compra, por lo que el checkout concentra la mayor fricción.
- **Retención:** estable en ~40–44 % semanal para las cohortes de enero a mayo de 2025.
- **A/B test del checkout:** +0.60 pp (p = 0.416), resultado **inconcluso**; el IC 95 % va de −0.84 a +2.03 pp.


---

# 🔹 Paso 1: Cargar y validar la calidad de los datos

---

# 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarse con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `orders_raw.csv`
  - `catalog.csv`
  - `marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [ ]:
# importar librerías
import numpy as np
import pandas as pd

In [ ]:
# Cargar archivos (datasets del curso; el enlace de descarga se omite. Colocar los CSV en la carpeta data/)
orders = pd.read_csv('data/orders_raw.csv')
catalog = pd.read_csv('data/catalog.csv')
marketing = pd.read_csv('data/marketing_spend.csv')

In [ ]:
# explorar dataset orders
orders.info()
orders.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [ ]:
# Explorar valores nulos en orders
print('Conteo nulos')
print(orders.isna().sum())
print('Porcentaje nulos')
print((orders.isna().sum() / len(orders) * 100).round(2))

Conteo nulos
id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     80
cantidad               50
precio_unitario        50
monto_descuento        50
monto_total             0
dtype: int64
Porcentaje nulos
id_pedido             0.00
id_usuario            0.00
fecha_hora_pedido     0.00
pais                  1.20
dispositivo           0.08
fuente_referencia     0.12
nombre_producto       0.12
categoria_producto    0.32
cantidad              0.20
precio_unitario       0.20
monto_descuento       0.20
monto_total           0.00
dtype: float64


In [ ]:
# explorar dataset catalog
catalog.info()
catalog.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 356.0+ bytes


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [ ]:
#explorar valores nulos en Catalog
print(catalog.isna().sum())

nombre_producto       0
categoria_producto    0
costo_unitario        0
proveedor             0
dtype: int64


In [ ]:
#Explorar dataset marketing
marketing.info()
marketing.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


In [ ]:
# Explorar valores nulos en marketing
print('Conteo nulos')
print(marketing.isna().sum())
print('Porcentaje nulos')
print((marketing.isna().sum() / len(marketing) * 100).round(2))

Conteo nulos
fecha           0
pais            0
id_campaña      0
canal         101
gasto           0
dtype: int64
Porcentaje nulos
fecha         0.00
pais          0.00
id_campaña    0.00
canal         6.23
gasto         0.00
dtype: float64


In [ ]:
marketing ['canal'].value_counts(dropna=False)

,count
canal,
paid_search,507
organic,506
social,506
NaN,101


# Resultados de la primera visualización
- **orders:** El dataframe se cargó con éxito. Se identificaron valores nulos en las columnas país, dispositivo, fuente_de_referencia, nombre_del_producto,categoría_del_producto, cantidad, precio_unitario y monto_descuento. Sin embargo, el porcentaje de nulos identificados por columna es bajo (país 1.2%, categoría 0.3%, el resto < 0.2%).

- **catalog:** El dataframe se cargó con éxito y, en primera instancia, no presenta valores nulos.

- **marketing:** El dataframe se cargó con éxito; únicamente se reportan valores nulos en la columna canal, los cuales deben ser revisados.

# Generalidades
- Las columnas de fechas se cargaron con tipo de dato object, por lo que deben ser transformadas a datetime.
- **marketing:** `canal` tiene ≈6.2% de nulos (101 de 1,620).

# Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas

---

In [ ]:
# bitácora de limpieza una fila por regla aplicada a orders
bitacora = []
n_inicial = len(orders)

def registrar(regla,antes, despues):
  bitacora.append({
      'regla':regla,
      'antes':antes,
      'despues':despues,
      'filas_eliminadas' : antes - despues,
      'pct_del_total_inicial': round ((antes - despues) / n_inicial * 100, 3)
      })

In [ ]:
# Convertir la columna de fecha en orders y marketing a tipo fecha
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'], errors='coerce')
marketing ['fecha'] = pd.to_datetime(marketing ['fecha'],errors='coerce')
# verificar el cambio
print (orders.info())
print (marketing.info())
# verificación adicional
print('Fechas no convertibles (NaT) en orders:', orders['fecha_hora_pedido'].isna().sum())
print('Fechas no convertibles (NaT) en marketing:', marketing['fecha'].isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25100 non-null  object        
 1   id_usuario          25100 non-null  object        
 2   fecha_hora_pedido   25100 non-null  datetime64[ns]
 3   pais                24800 non-null  object        
 4   dispositivo         25080 non-null  object        
 5   fuente_referencia   25070 non-null  object        
 6   nombre_producto     25070 non-null  object        
 7   categoria_producto  25020 non-null  object        
 8   cantidad            25050 non-null  float64       
 9   precio_unitario     25050 non-null  float64       
 10  monto_descuento     25050 non-null  float64       
 11  monto_total         25100 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.3+ MB
None
<class 'pandas.core.frame.DataF

---
# Revisión columnas numéricas.

In [ ]:
#Revisión columnas numéricas en orders
print(orders.describe())
print('Conteo valores en cantidad')
print(orders['cantidad'].value_counts(dropna=False))
print('Conteo valores negativos en  monto total')
print((orders['monto_total']< 0).sum())

                   fecha_hora_pedido      cantidad  precio_unitario  \
count                          25100  25050.000000     25050.000000   
mean   2025-04-01 03:21:22.231075584      7.092735       259.305549   
min              2025-01-01 00:00:00     -2.000000        20.030000   
25%              2025-02-15 00:00:00      1.000000       138.377500   
50%              2025-04-02 00:00:00      2.000000       258.715000   
75%              2025-05-16 00:00:00      2.000000       380.332500   
max              2025-06-30 00:00:00  20000.000000       499.960000   
std                              NaN    296.277003       138.726461   

       monto_descuento   monto_total  
count     25050.000000  2.510000e+04  
mean          4.500798  2.072680e+03  
min           0.000000 -4.926500e+02  
25%           0.000000  1.805075e+02  
50%           0.000000  3.417500e+02  
75%          10.000000  5.185800e+02  
max          15.000000  8.840200e+06  
std           5.223010  9.894995e+04  
Conteo va

In [ ]:
# Revisión detallada nulos en cantidad
orders[orders['cantidad'].isnull()]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
74,order_74,user_6172,2025-01-15,Argentina,desktop,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,595.85
75,order_75,user_6588,2025-06-19,Colombia,mobile,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,458.15
76,order_76,user_3193,2025-03-17,Argentina,mobile,paid_search,Laptop-Gaming-16GB,NaN,NaN,NaN,NaN,319.75
77,order_77,user_775,2025-06-24,Argentina,desktop,social,Vacuum-Pro-Black,NaN,NaN,NaN,NaN,227.55
78,order_78,user_2702,2025-03-19,Argentina,desktop,paid_search,Phone-Pro-128GB,NaN,NaN,NaN,NaN,432.39
79,order_79,user_6438,2025-03-22,Colombia,desktop,paid_search,Sneakers-Urban-42,NaN,NaN,NaN,NaN,527.15
80,order_80,user_6790,2025-01-03,Mexico,desktop,organic,Vacuum-Pro-Black,NaN,NaN,NaN,NaN,711.18
81,order_81,user_232,2025-05-16,Mexico,mobile,paid_search,Tablet-Standard-64GB,NaN,NaN,NaN,NaN,41.08
82,order_82,user_5665,2025-02-07,Colombia,mobile,paid_search,Sneakers-Urban-42,NaN,NaN,NaN,NaN,431.55
83,order_83,user_5225,2025-04-25,Argentina,mobile,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,279.35


In [ ]:
# Eliminar filas con valores nulos en cantidad, precio_unitario y monto descuento.
n0 = len(orders)
orders = orders.dropna(subset=['cantidad','precio_unitario', 'monto_descuento']).reset_index(drop=True)
registrar('Nulos en cantidad/precio_unitario/monto_descuento', n0, len(orders))

In [ ]:
# Filtro para valores no positivos en cantidad
n0 = len(orders)
print('Filas con cantidad <= 0:', (orders['cantidad'] <= 0).sum())
orders = orders [orders['cantidad'] > 0].reset_index(drop=True)
registrar('Cantidad <= 0', n0, len(orders))

Filas con cantidad <= 0: 4


In [ ]:
# Inspeccionar la distribución ANTES de fijar un umbral de outliers
print(orders['cantidad'].value_counts().sort_index())
print(orders['cantidad'].describe(percentiles=[.5, .99, .999]))

cantidad
1.0        12394
2.0        12642
10000.0        6
20000.0        4
Name: count, dtype: int64
count    25046.000000
mean         7.094067
std        296.300643
min          1.000000
50%          2.000000
99%          2.000000
99.9%        2.000000
max      20000.000000
Name: cantidad, dtype: float64


In [ ]:
# umbral de outliers derivado de los datos (Q3 + 3*IQR) y aplicado sobre `cantidad`
q1, q3 = orders['cantidad'].quantile([.25, .75])
limite_cantidad = q3 + 3 * (q3 - q1)
print('Límite superior de cantidad (Q3 + 3*IQR):', limite_cantidad)

n0 = len(orders)
print('Outliers detectados:', (orders['cantidad'] > limite_cantidad).sum())
orders = orders[orders['cantidad'] <= limite_cantidad].reset_index(drop=True)
registrar('Outliers en cantidad (> Q3 + 3*IQR)', n0, len(orders))

# Revisión: tras lo anterior no debe quedar ningún monto_total <= 0
n0 = len(orders)
print('monto_total <= 0 restantes:', (orders['monto_total'] <= 0).sum())
orders = orders[orders['monto_total'] > 0].reset_index(drop=True)
registrar('monto_total <= 0 (guarda)', n0, len(orders))
print('monto_total máximo tras limpieza:', orders['monto_total'].max())


Límite superior de cantidad (Q3 + 3*IQR): 5.0
Outliers detectados: 10
monto_total <= 0 restantes: 0
monto_total máximo tras limpieza: 999.89


In [ ]:
# Revisión columnas numéricas en orders después de tratamiento
print(orders.describe())
print('Conteo valores en cantidad')
print(orders['cantidad'].value_counts(dropna=False))
print('Conteo valores negativos en  monto_total')
print((orders['monto_total']< 0).sum())

                   fecha_hora_pedido      cantidad  precio_unitario  \
count                          25036  25036.000000     25036.000000   
mean   2025-04-01 03:17:30.838792192      1.504953       259.293912   
min              2025-01-01 00:00:00      1.000000        20.030000   
25%              2025-02-15 00:00:00      1.000000       138.392500   
50%              2025-04-02 00:00:00      2.000000       258.700000   
75%              2025-05-16 00:00:00      2.000000       380.297500   
max              2025-06-30 00:00:00      2.000000       499.960000   
std                              NaN      0.499985       138.716939   

       monto_descuento   monto_total  
count     25036.000000  25036.000000  
mean          4.502516    385.807512  
min           0.000000      5.240000  
25%           0.000000    180.330000  
50%           0.000000    341.360000  
75%          10.000000    517.640000  
max          15.000000    999.890000  
std           5.223500    255.750599  
Conteo va

---
# Resultados de revisión y tratamiento
- **Valores no positivos:** Se identificaron cuatro valores negativos en `cantidad` y su `monto_total`asociados. Se eliminaron por representan errores de captura.

- **Valores faltantes (NaN):** 50 registros nulos coincidían en `cantidad`, `precio_unitario` y `monto_descuento`. Se eliminaron por su baja representatividad (≈0.2%) y por no existir un criterio confiable para su imputación.

- **Valores atípicos (outliers):** 10 registros con `cantidad` de 10,000 y 20,000, frente a valores habituales de 1 y 2. El umbral se calculó con la regla Q3 + 3·IQR sobre `cantidad`. Se eliminaron por no pertenecer a la distribución habitual.


---
# Revisión columnas numéricas en Catalog y Marketing

In [ ]:
# Revisión columnas numéricas en catalog
print(catalog.describe())
print('Conteo valores en costo unitario')
print(catalog ['costo_unitario'].value_counts(dropna=False))

       costo_unitario
count        7.000000
mean       102.252857
std        111.011563
min         10.120000
25%         16.905000
50%         25.210000
75%        182.975000
max        280.680000
Conteo valores en costo unitario
costo_unitario
280.68    1
10.12     1
25.21     1
176.64    1
16.60     1
17.21     1
189.31    1
Name: count, dtype: int64


In [ ]:
# Revisión Columnas numéricas en marketing
print(marketing.describe())
print('Conteo valores en gasto')
print(marketing['gasto'].value_counts(dropna=False))

                     fecha       gasto
count                 1620  1620.00000
mean   2025-03-31 12:00:00  1772.74292
min    2025-01-01 00:00:00   501.11000
25%    2025-02-14 18:00:00  1128.03000
50%    2025-03-31 12:00:00  1782.42500
75%    2025-05-15 06:00:00  2420.68500
max    2025-06-29 00:00:00  2999.36000
std                    NaN   734.43294
Conteo valores en gasto
gasto
1018.16    2
2063.86    2
1523.78    2
619.11     2
2696.62    2
          ..
2565.67    1
1285.76    1
2303.14    1
1677.61    1
2220.58    1
Name: count, Length: 1612, dtype: int64


---
# Resultados
- **Catalog y Marketing:** No se identificaron valores negativos ni outliers en las variables numéricas. Asimismo, se descartó la presencia de valores nulos (NaN) implícitos o camuflados en ambos dataframes.

---
# Consistencia de monto total en orders

In [ ]:
# Verificar consistencia de monto
orders ['total_esperado'] = orders ['cantidad'] * orders ['precio_unitario'] - orders ['monto_descuento']
diferencia = abs(orders ['monto_total'] - orders ['total_esperado'])
inconsistencias = orders [diferencia > 0.1]
print('Filas inconsistentes:', len(inconsistencias))
orders = orders.drop(columns='total_esperado')

Filas inconsistentes: 0


---
# Resultado
- Los montos totales reportados en orders son consistentes.

---
# Revisión duplicados

In [ ]:
# Revisión general de duplicados
print ('Duplicados en orders')
print (orders.duplicated().sum())
print ('Duplicados en catalogo')
print (catalog.duplicated().sum())
print ('Duplicados en marketing')
print(marketing.duplicated().sum())

Duplicados en orders
100
Duplicados en catalogo
0
Duplicados en marketing
0


In [ ]:
# Revisión específica duplicados en orders
keys = ["id_pedido","fecha_hora_pedido"]
orders.duplicated(subset=keys).sum()

np.int64(100)

In [ ]:
# Eliminación de duplicados
n0 = len(orders)
orders = orders.drop_duplicates().reset_index(drop=True)
registrar('Duplicados exactos', n0, len(orders))
print('Duplicados exactos restantes:', orders.duplicated().sum())

Duplicados exactos restantes: 0


---

# Resultado y tratamiento
- **Registros duplicados:** Se identificaron 100 filas duplicadas. Tras verificar su coincidencia en la clave primaria (id_pedido) y la marca temporal (fecha_hora_pedido), se procedió a eliminar los duplicados del dataframe orders.


---
# Revisión columnas Categóricas

In [ ]:
# Función para revisar variables categóricas identificadas en cada dataframe
def revisar_categorica(df, columna, dropna=False):
    return df[columna].value_counts(dropna=dropna)

---
# Revisión columnas Categóricas en orders

In [ ]:
# Revisar columnas categóricas en orders
print(revisar_categorica (orders, 'pais'))
print(revisar_categorica (orders, 'dispositivo'))
print(revisar_categorica (orders, 'fuente_referencia'))
print(revisar_categorica (orders, 'nombre_producto'))
print(revisar_categorica (orders, 'categoria_producto'))

pais
Colombia     7465
Mexico       7461
Argentina    7236
mexico        861
colombia      822
argentina     795
NaN           296
Name: count, dtype: int64
dispositivo
desktop    12676
mobile     12240
NaN           20
Name: count, dtype: int64
fuente_referencia
social         8382
organic        8267
paid_search    8257
NaN              30
Name: count, dtype: int64
nombre_producto
Blender-XL-Red          4176
Vacuum-Pro-Black        4170
Jacket-Winter-M         4166
Sneakers-Urban-42       4129
Laptop-Gaming-16GB      2768
Tablet-Standard-64GB    2764
Phone-Pro-128GB         2733
NaN                       30
Name: count, dtype: int64
categoria_producto
Hogar          8346
Moda           8295
Electronica    8265
NaN              30
Name: count, dtype: int64


In [ ]:
# Normalizar e imputar Columna País
orders['pais'] = (orders['pais'].str.strip().str.title().fillna('Sin datos').astype('category'))
print(orders['pais'].value_counts())
print('Conteo nulos')
print(orders.isna().sum())

pais
Mexico       8322
Colombia     8287
Argentina    8031
Sin datos     296
Name: count, dtype: int64
Conteo nulos
id_pedido              0
id_usuario             0
fecha_hora_pedido      0
pais                   0
dispositivo           20
fuente_referencia     30
nombre_producto       30
categoria_producto    30
cantidad               0
precio_unitario        0
monto_descuento        0
monto_total            0
dtype: int64


In [ ]:
# Revisión especifica nulos faltantes en orders
orders[orders['nombre_producto'].isnull()]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
44,order_44,user_2899,2025-02-22,Colombia,desktop,NaN,NaN,NaN,1.0,318.38,5.0,313.38
45,order_45,user_6196,2025-02-10,Mexico,mobile,NaN,NaN,NaN,2.0,354.06,10.0,698.12
46,order_46,user_5815,2025-04-24,Mexico,mobile,NaN,NaN,NaN,1.0,359.31,10.0,349.31
47,order_47,user_406,2025-05-08,Colombia,mobile,NaN,NaN,NaN,2.0,476.78,0.0,953.56
48,order_48,user_6187,2025-01-26,Colombia,desktop,NaN,NaN,NaN,2.0,137.96,0.0,275.93
49,order_49,user_7225,2025-04-04,Argentina,desktop,NaN,NaN,NaN,2.0,433.76,0.0,867.53
50,order_50,user_1840,2025-02-05,Mexico,desktop,NaN,NaN,NaN,1.0,495.38,5.0,490.38
51,order_51,user_5998,2025-02-21,Mexico,desktop,NaN,NaN,NaN,2.0,382.94,0.0,765.89
52,order_52,user_5679,2025-04-17,Argentina,mobile,NaN,NaN,NaN,2.0,415.01,0.0,830.01
53,order_53,user_5936,2025-01-02,Argentina,desktop,NaN,NaN,NaN,2.0,236.49,0.0,472.98


In [ ]:
# Eliminar nulos en columna nombre_producto
n0 = len(orders)
orders = orders.dropna(subset=['nombre_producto']).reset_index(drop=True)
registrar('Nulos en nombre_producto (coinciden con fuente_referencia y categoria_producto)', n0, len(orders))
print ('Conteo nulos')
print (orders.isna().sum())

Conteo nulos
id_pedido              0
id_usuario             0
fecha_hora_pedido      0
pais                   0
dispositivo           20
fuente_referencia      0
nombre_producto        0
categoria_producto     0
cantidad               0
precio_unitario        0
monto_descuento        0
monto_total            0
dtype: int64


In [ ]:
# Marcar valores nulos en dispositivo
orders['dispositivo'] = orders['dispositivo'].str.strip().str.lower().fillna('Sin datos')
print(orders['dispositivo'].value_counts())
print(orders.info())

dispositivo
desktop      12661
mobile       12225
Sin datos       20
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24906 entries, 0 to 24905
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           24906 non-null  object        
 1   id_usuario          24906 non-null  object        
 2   fecha_hora_pedido   24906 non-null  datetime64[ns]
 3   pais                24906 non-null  category      
 4   dispositivo         24906 non-null  object        
 5   fuente_referencia   24906 non-null  object        
 6   nombre_producto     24906 non-null  object        
 7   categoria_producto  24906 non-null  object        
 8   cantidad            24906 non-null  float64       
 9   precio_unitario     24906 non-null  float64       
 10  monto_descuento     24906 non-null  float64       
 11  monto_total         24906 non-null  float64       
dtypes: cate

---
# Resultados y tratamiento
- **País:** Se corrigieron inconsistencias de mayúsculas/minúsculas(`mexico`, `colombia`, `argentina`). Los ≈1.2% de nulos (296 registros) se categorizaron como **"Sin datos"** para conservar las demás variables de esas filas. Esta categoría se excluye de los análisis por país (ROAS por país).

- **fuente_de_referencia, nombre_producto y categoria_producto:** los 30 nulos coincidían entre las tres columnas. Al no existir un método confiable de imputación y dado su bajo porcentaje, se eliminaron esas filas.

- **dispositivo:** Los 20 nulos se imputaron como **"Sin datos"** para no perder registros con variables numéricas válidas.  

---
# Revisión columnas Categóricas en catalog y marketing

In [ ]:
# Revisar columnas categóricas en catalogo
print(revisar_categorica (catalog, 'nombre_producto'))
print(revisar_categorica (catalog, 'categoria_producto'))
print(revisar_categorica (catalog, 'proveedor'))

nombre_producto
Laptop-Gaming-16GB      1
Phone-Pro-128GB         1
Tablet-Standard-64GB    1
Blender-XL-Red          1
Vacuum-Pro-Black        1
Sneakers-Urban-42       1
Jacket-Winter-M         1
Name: count, dtype: int64
categoria_producto
Electrónica    3
Hogar          2
Moda           2
Name: count, dtype: int64
proveedor
Fuller, Pena and Myers     1
King Ltd                   1
Bowers LLC                 1
Long-Reid                  1
Rivera, Carr and Finley    1
Greene-Smith               1
Mcmillan-Rhodes            1
Name: count, dtype: int64


In [ ]:
# Ajustar consistencia en categoría de producto
catalog['categoria_producto'] = (catalog['categoria_producto']
    .str.strip().str.normalize('NFKD')
    .str.encode('ascii', errors='ignore').str.decode('utf-8'))

In [ ]:
# Revisar columnas categóricas en marketing
print (revisar_categorica (marketing, 'pais'))
print (revisar_categorica (marketing, 'canal'))
print (revisar_categorica (marketing, 'id_campaña'))

pais
Mexico       540
Colombia     540
Argentina    540
Name: count, dtype: int64
canal
paid_search    507
organic        506
social         506
NaN            101
Name: count, dtype: int64
id_campaña
organic_Mexico           180
paid_search_Mexico       180
social_Mexico            180
organic_Colombia         180
paid_search_Colombia     180
social_Colombia          180
organic_Argentina        180
paid_search_Argentina    180
social_Argentina         180
Name: count, dtype: int64


In [ ]:
canales_por_campana = marketing.groupby('id_campaña')['canal'].nunique()
print(canales_por_campana)

id_campaña
organic_Argentina        1
organic_Colombia         1
organic_Mexico           1
paid_search_Argentina    1
paid_search_Colombia     1
paid_search_Mexico       1
social_Argentina         1
social_Colombia          1
social_Mexico            1
Name: canal, dtype: int64


In [ ]:
mapeo = marketing.groupby('id_campaña')['canal'].first()
marketing['canal'] = marketing['canal'].fillna(marketing ['id_campaña'].map(mapeo))
print(marketing.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1620 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 63.4+ KB
None


---
# Resultados y tratamiento
- **catalog:** Las columnas categóricas son consistentes y no se identifican valores nulos. Se realizó un ajuste de estandarización en la columna categoría_producto para unificar los nombres de las categorías.

- **marketing:** Se identificaron únicamente 101 valores nulos en la columna canal, los cuales fueron imputados a través de la columna id_campaña, ya que esta permite determinar el canal correspondiente.  


In [ ]:
# Resumen de la limpieza de orders
bitacora_df = pd.DataFrame(bitacora)
display(bitacora_df)
print('Filas iniciales:', n_inicial, '| finales:', len(orders),
      '| eliminadas:', n_inicial - len(orders), f'({(n_inicial - len(orders)) / n_inicial:.2%})')
assert bitacora_df['filas_eliminadas'].sum() == n_inicial - len(orders)

,regla,antes,despues,filas_eliminadas,pct_del_total_inicial
0,Nulos en cantidad/precio_unitario/monto_descuento,25100,25050,50,0.199
1,Cantidad <= 0,25050,25046,4,0.016
2,Outliers en cantidad (> Q3 + 3*IQR),25046,25036,10,0.040
3,monto_total <= 0 (guarda),25036,25036,0,0.000
4,Duplicados exactos,25036,24936,100,0.398
5,Nulos en nombre_producto (coinciden con fuente...,24936,24906,30,0.120


Filas iniciales: 25100 | finales: 24906 | eliminadas: 194 (0.77%)


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [ ]:
# Exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

# 🔹 Paso 2: Analizar si el negocio es rentable

## 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)?
- ¿Cuál es el costo total?
- ¿Cuánto se ha invertido en marketing?
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden?
- ¿Cuál es la cantidad promedio de productos por orden?
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal?

In [ ]:
# validar las llaves antes del merge del Paso 2
assert catalog['nombre_producto'].is_unique, 'catalog.nombre_producto no es único: el merge duplicaría filas'
faltan = set(orders['nombre_producto']) - set(catalog['nombre_producto'])
print('Productos de orders sin costo en catálogo:', faltan if faltan else 'ninguno')
print('Categorías orders :', sorted(orders['categoria_producto'].dropna().unique()))
print('Categorías catalog:', sorted(catalog['categoria_producto'].unique()))

Productos de orders sin costo en catálogo: ninguno
Categorías orders : ['Electronica', 'Hogar', 'Moda']
Categorías catalog: ['Electronica', 'Hogar', 'Moda']


In [ ]:
# Costo:Merge validado.
merged = orders.merge(catalog[['nombre_producto', 'costo_unitario']],
                      on='nombre_producto', how='left', validate='many_to_one')
assert merged['costo_unitario'].notna().all(), 'Hay pedidos sin costo en el catálogo'
assert len(merged) == len(orders), 'El merge cambió el número de filas'
merged['costo'] = merged['costo_unitario'] * merged['cantidad']

In [ ]:
# Rentabilidad del negocio
# Revenue, costo y margen bruto
revenue_total = orders['monto_total'].sum()
costo_total = merged['costo'].sum()
margen_bruto = revenue_total - costo_total

# Marketing: solo el periodo cubierto por los pedidos
f_ini = orders['fecha_hora_pedido'].min().normalize()
f_fin = orders['fecha_hora_pedido'].max()
marketing_periodo = marketing[marketing['fecha'].between(f_ini, f_fin)]
inv_mark = marketing_periodo['gasto'].sum()

# Utilidad después de marketing (margen bruto - inversión en marketing)
utilidad_post_marketing = margen_bruto - inv_mark

# Mostrar resultados
print('Revenue Total              :', round(revenue_total, 2))
print('Costo Total                :', round(costo_total, 2))
print('Margen bruto               :', round(margen_bruto, 2), f'({margen_bruto / revenue_total:.1%} del revenue)')
print('Inversión Marketing        :', round(inv_mark, 2))
print('Utilidad post-marketing    :', round(utilidad_post_marketing, 2), f'({utilidad_post_marketing / revenue_total:.1%} del revenue)')
print('ROAS global                :', round(revenue_total / inv_mark, 2))

Revenue Total              : 9610018.94
Costo Total                : 3828869.01
Margen bruto               : 5781149.93 (60.2% del revenue)
Inversión Marketing        : 2871843.53
Utilidad post-marketing    : 2909306.4 (30.3% del revenue)
ROAS global                : 3.35


In [ ]:
# Métricas por pedido + rentabilidad por producto.
ventas_por_pedido = orders.groupby('id_pedido')['monto_total'].sum()
ticket_promedio = ventas_por_pedido.mean()
cantidad_prom_productos = orders.groupby('id_pedido')['cantidad'].sum().mean()

tabla_producto = merged.groupby('nombre_producto').agg(
    unidades=('cantidad', 'sum'), revenue=('monto_total', 'sum'), costo=('costo', 'sum'))
tabla_producto['margen_bruto'] = tabla_producto['revenue'] - tabla_producto['costo']
tabla_producto['margen_%'] = (tabla_producto['margen_bruto'] / tabla_producto['revenue'] * 100).round(1)
producto_top = tabla_producto['unidades'].idxmax()
producto_mas_rentable = tabla_producto['margen_bruto'].idxmax()

print('Ticket promedio por pedido :', round(ticket_promedio, 2))
print('Productos promedio/pedido  :', round(cantidad_prom_productos, 2))
print('Producto más vendido (unidades)  :', producto_top)
print('Producto con mayor margen bruto  :', producto_mas_rentable)
print('\nRentabilidad por producto')
print(tabla_producto.sort_values('margen_bruto', ascending=False).round(2))


Ticket promedio por pedido : 385.85
Productos promedio/pedido  : 1.5
Producto más vendido (unidades)  : Vacuum-Pro-Black
Producto con mayor margen bruto  : Vacuum-Pro-Black

Rentabilidad por producto
                      unidades     revenue       costo  margen_bruto  margen_%
nombre_producto                                                               
Vacuum-Pro-Black        6284.0  1621051.85   104314.40    1516737.45      93.6
Sneakers-Urban-42       6172.0  1566075.93   106220.12    1459855.81      93.2
Phone-Pro-128GB         4140.0  1057134.62    41896.80    1015237.82      96.0
Tablet-Standard-64GB    4153.0  1062922.80   104697.13     958225.67      90.2
Blender-XL-Red          6279.0  1611574.59  1109122.56     502452.03      31.2
Jacket-Winter-M         6256.0  1606586.17  1184323.36     422262.81      26.3
Laptop-Gaming-16GB      4198.0  1084672.98  1178294.64     -93621.66      -8.6


In [ ]:
# Metricas de Marketing por canal y pais
gasto_canal_marke = marketing_periodo.groupby('canal')['gasto'].sum()
print('\nGasto marketing por canal')
print(gasto_canal_marke.round(2))

# ROAS por canal (fuente_referencia de orders ↔ canal de marketing)
roas_canal = pd.DataFrame({'revenue': orders.groupby('fuente_referencia')['monto_total'].sum(),
                           'gasto': gasto_canal_marke})
roas_canal['ROAS'] = (roas_canal['revenue'] / roas_canal['gasto']).round(2)
print('\nROAS por canal'); print(roas_canal.round(2))

# ROAS por país (se excluye 'Sin datos')
gasto_pais = (marketing_periodo.assign(pais=marketing_periodo['pais'].str.strip().str.title())
              .groupby('pais')['gasto'].sum())
roas_pais = pd.DataFrame({'revenue': orders.groupby(orders['pais'].astype(str))['monto_total'].sum(),
                          'gasto': gasto_pais}).dropna()
roas_pais['ROAS'] = (roas_pais['revenue'] / roas_pais['gasto']).round(2)
print('\nROAS por país'); print(roas_pais.round(2))



Gasto marketing por canal
canal
organic        972650.96
paid_search    922374.20
social         976818.37
Name: gasto, dtype: float64

ROAS por canal
                revenue      gasto  ROAS
organic      3178228.55  972650.96  3.27
paid_search  3209280.07  922374.20  3.48
social       3222510.32  976818.37  3.30

ROAS por país
              revenue      gasto  ROAS
pais                                  
Argentina  3120649.33  947694.60  3.29
Colombia   3170333.82  935653.42  3.39
Mexico     3207047.74  988495.51  3.24


# 🔹 Paso 2: Resultados Análisis de rentabilidad del negocio

## KPIs principales
**📊 Parte 1: Rentabilidad del negocio** (primera mitad de 2025)

- Ingreso total: **9,610,018.94**
- Costo de producto: **3,828,869.01**
- Margen bruto: **5,781,149.93** (60.2% del revenue)
- Inversión en marketing del periodo: **2,871,843.53**
- **utilidad después de marketing:** **2,909,306.40** (30.3% del revenue)
- **ROAS global:** **3.35**

---

**📈 Parte 2: Comportamiento de ventas**
- Ticket promedio por pedido: **385.85**
- Productos por pedido: **1.5**
- Producto más vendido: **Vacuum-Pro-Black**
- Gasto en marketing por canal: social **976,818.37** · organic  **972,650.96** · paid search **922,374.20**
- ROAS por canal: social **3.30** · organic  **3.27** · paid search **3.48** · ROAS por país: Argentina **3.29** · Colombia **3.39** · Mexico **3.24**

✍️ **Interpretación:**
- Rentabilidad global: el comercio electrónico generó ingresos por 9,610,018.94 con un margen bruto de 60.2 %. Tras descontar la inversión en marketing (2,871,843.53), la utilidad fue de 2,909,306.40, equivalente al 30.3 % de los ingresos. Es un resultado positivo, aunque todavía no incluye costos logísticos, tributarios ni de operación. El ROAS global de 3.35 supera ampliamente el punto de equilibrio (≈ 1.66, es decir, 1 / margen bruto de 60.2 %).
- Productos: Vacuum-Pro-Black lidera en unidades vendidas (6,284) y en margen bruto (1,516,737.45). Phone-Pro-128GB tiene el mayor margen porcentual (96.0 %), aunque es el de menor volumen (4,140 unidades). Se observa una brecha marcada: cuatro productos superan el 90 % de margen, mientras que Blender-XL-Red (31.2 %), Jacket-Winter-M (26.3 %) y Laptop-Gaming-16GB (−8.6 %) rinden muy por debajo. La laptop genera un margen bruto negativo (−93,621.66), por lo que se recomienda revisar su precio y costo con prioridad, seguida de la chaqueta y la licuadora.

- Marketing: la inversión está distribuida de forma pareja entre canales y países. El rendimiento es homogéneo, con ROAS entre 3.27 y 3.48 por canal y entre 3.24 y 3.39 por país. paid_search (3.48) y Colombia (3.39) muestran el mejor ROAS con el menor gasto relativo, por lo que podrían evaluarse como candidatos para aumentar inversión de forma gradual.

---

# 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

> 🔒 **Nota:** las credenciales de la base (acceso restringido del curso) fueron omitidas y se leen desde variables de entorno (`DB_USER`, `DB_PWD`, `DB_HOST`, `DB_NAME`). Las celdas SQL conservan las salidas ejecutadas originalmente; sin acceso a la base no pueden volver a ejecutarse.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [ ]:
!pip install -q psycopg2-binary psycopg

import os
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión a la base de datos del curso (acceso restringido)
# Las credenciales NO se incluyen en este notebook: se leen de variables de entorno.
#   DB_USER, DB_PWD, DB_HOST, DB_NAME (y opcionalmente DB_PORT)
# Las celdas SQL siguientes conservan las salidas ejecutadas originalmente.
# ======================
db_config = {
    'user': os.getenv('DB_USER'),
    'pwd': os.getenv('DB_PWD'),
    'host': os.getenv('DB_HOST'),
    'port': int(os.getenv('DB_PORT', 5432)),
    'db': os.getenv('DB_NAME')
}

faltantes = [k for k in ('user', 'pwd', 'host', 'db') if not db_config[k]]
if faltantes:
    print('Credenciales no configuradas (' + ', '.join(faltantes) + '): las celdas SQL no se pueden ejecutar; ver salidas guardadas.')

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'], db_config['pwd'], db_config['host'], db_config['port'], db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode': 'require'})

In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [ ]:
# PARTE 1: Totales de Usuarios por Etapa
# ======================
query_totals = '''
WITH cte_visit AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'first_visit'
),
  cte_select AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'select_item'
),
cte_cart AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'add_to_cart'
),
cte_checkout AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'begin_checkout'
),
cte_payment AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'add_payment_info'
),
cte_purchase AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'purchase'
)

-- SELECT final con conteos de cada etapa
SELECT
(SELECT COUNT (*) FROM cte_visit) AS first_visit_users,
(SELECT COUNT (*) FROM cte_select) AS select_item_users,
(SELECT COUNT (*) FROM cte_cart) AS add_to_cart_users,
(SELECT COUNT (*) FROM cte_checkout) AS begin_checkout_users,
(SELECT COUNT (*) FROM cte_payment) AS add_payment_users,
(SELECT COUNT (*) FROM cte_purchase) AS purchase_users;
'''

totals = pd.read_sql(query_totals, con=engine)
totals


,first_visit_users,select_item_users,add_to_cart_users,begin_checkout_users,add_payment_users,purchase_users
0,7796,7582,7634,7208,6250,6240


In [ ]:
# PARTE 2: Conversiones Funnel sin orden
# ======================

query_conversion = '''
WITH cte_visit AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'first_visit'
),
cte_select AS (
  SELECT DISTINCT e.id_usuario
  FROM events e
  INNER JOIN cte_visit v ON e.id_usuario = v.id_usuario
  WHERE e.nombre_evento = 'select_item'
),
cte_cart AS (
  SELECT DISTINCT e.id_usuario
  FROM events e
  INNER JOIN cte_select s ON e.id_usuario = s.id_usuario
  WHERE e.nombre_evento = 'add_to_cart'
),
cte_checkout AS (
  SELECT DISTINCT e.id_usuario
  FROM events e
  INNER JOIN cte_cart c ON e.id_usuario = c.id_usuario
  WHERE e.nombre_evento = 'begin_checkout'
),
cte_payment AS (
  SELECT DISTINCT e.id_usuario
  FROM events e
  INNER JOIN cte_checkout ch ON e.id_usuario = ch.id_usuario
  WHERE e.nombre_evento = 'add_payment_info'
),
cte_purchase AS (
  SELECT DISTINCT e.id_usuario
  FROM events e
  INNER JOIN cte_payment p ON e.id_usuario = p.id_usuario
  WHERE e.nombre_evento = 'purchase'
)

-- SELECT final con conteos de cada etapa
SELECT
  (SELECT COUNT(*) FROM cte_visit)    AS first_visit_users,
  (SELECT COUNT(*) FROM cte_select)   AS select_item_users,
  (SELECT COUNT(*) FROM cte_cart)     AS add_to_cart_users,
  (SELECT COUNT(*) FROM cte_checkout) AS begin_checkout_users,
  (SELECT COUNT(*) FROM cte_payment)  AS add_payment_users,
  (SELECT COUNT(*) FROM cte_purchase) AS purchase_users,

  ((SELECT COUNT(*) FROM cte_visit) - (SELECT COUNT(*) FROM cte_select)) * 100.0
    / NULLIF((SELECT COUNT(*) FROM cte_visit), 0)    AS dropoff_after_first_visit_pct,
  ((SELECT COUNT(*) FROM cte_select) - (SELECT COUNT(*) FROM cte_cart)) * 100.0
    / NULLIF((SELECT COUNT(*) FROM cte_select), 0)   AS dropoff_after_select_item_pct,
  ((SELECT COUNT(*) FROM cte_cart) - (SELECT COUNT(*) FROM cte_checkout)) * 100.0
    / NULLIF((SELECT COUNT(*) FROM cte_cart), 0)     AS dropoff_after_add_to_cart_pct,
  ((SELECT COUNT(*) FROM cte_checkout) - (SELECT COUNT(*) FROM cte_payment)) * 100.0
    / NULLIF((SELECT COUNT(*) FROM cte_checkout), 0) AS dropoff_after_begin_checkout_pct,
  ((SELECT COUNT(*) FROM cte_payment) - (SELECT COUNT(*) FROM cte_purchase)) * 100.0
    / NULLIF((SELECT COUNT(*) FROM cte_payment), 0)  AS dropoff_after_add_payment_info_pct,
(SELECT COUNT(*) FROM cte_purchase) * 100.0
    / NULLIF((SELECT COUNT(*) FROM cte_visit), 0) AS conversion_rate_pct;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,first_visit_users,select_item_users,add_to_cart_users,begin_checkout_users,add_payment_users,purchase_users,dropoff_after_first_visit_pct,dropoff_after_select_item_pct,dropoff_after_add_to_cart_pct,dropoff_after_begin_checkout_pct,dropoff_after_add_payment_info_pct,conversion_rate_pct
0,7796,7393,7052,6364,4967,3857,5.169318,4.612471,9.756098,21.951603,22.347493,49.474089


In [ ]:
# Parte 2.1 Funnel estricto y comparación funnels
query_seq = '''
WITH fechas AS (
  SELECT DISTINCT id_usuario, nombre_evento,
         CAST(timestamp_evento AS DATE) AS fecha
  FROM events
  WHERE nombre_evento IN ('first_visit','select_item','add_to_cart',
                          'begin_checkout','add_payment_info','purchase')
),
s1 AS (
  SELECT id_usuario, MIN(fecha) AS t1
  FROM fechas WHERE nombre_evento = 'first_visit' GROUP BY id_usuario
),
s2 AS (
  SELECT s1.id_usuario, s1.t1, MIN(f.fecha) AS t2
  FROM s1 LEFT JOIN fechas f
    ON f.id_usuario = s1.id_usuario AND f.nombre_evento = 'select_item' AND f.fecha >= s1.t1
  GROUP BY s1.id_usuario, s1.t1
),
s3 AS (
  SELECT s2.id_usuario, s2.t1, s2.t2, MIN(f.fecha) AS t3
  FROM s2 LEFT JOIN fechas f
    ON f.id_usuario = s2.id_usuario AND f.nombre_evento = 'add_to_cart' AND f.fecha >= s2.t2
  GROUP BY s2.id_usuario, s2.t1, s2.t2
),
s4 AS (
  SELECT s3.id_usuario, s3.t1, s3.t2, s3.t3, MIN(f.fecha) AS t4
  FROM s3 LEFT JOIN fechas f
    ON f.id_usuario = s3.id_usuario AND f.nombre_evento = 'begin_checkout' AND f.fecha >= s3.t3
  GROUP BY s3.id_usuario, s3.t1, s3.t2, s3.t3
),
s5 AS (
  SELECT s4.id_usuario, s4.t1, s4.t2, s4.t3, s4.t4, MIN(f.fecha) AS t5
  FROM s4 LEFT JOIN fechas f
    ON f.id_usuario = s4.id_usuario AND f.nombre_evento = 'add_payment_info' AND f.fecha >= s4.t4
  GROUP BY s4.id_usuario, s4.t1, s4.t2, s4.t3, s4.t4
),
s6 AS (
  SELECT s5.id_usuario, s5.t1, s5.t2, s5.t3, s5.t4, s5.t5, MIN(f.fecha) AS t6
  FROM s5 LEFT JOIN fechas f
    ON f.id_usuario = s5.id_usuario AND f.nombre_evento = 'purchase' AND f.fecha >= s5.t5
  GROUP BY s5.id_usuario, s5.t1, s5.t2, s5.t3, s5.t4, s5.t5
)
SELECT COUNT(t1) AS first_visit_users, COUNT(t2) AS select_item_users,
       COUNT(t3) AS add_to_cart_users, COUNT(t4) AS begin_checkout_users,
       COUNT(t5) AS add_payment_users, COUNT(t6) AS purchase_users
FROM s6;
'''
funnel_seq = pd.read_sql(query_seq, con=engine)

etapas = ['first_visit', 'select_item', 'add_to_cart', 'begin_checkout', 'add_payment_info', 'purchase']
tabla_funnel = pd.DataFrame({
    'etapa': etapas,
    'usuarios_alcance': totals.iloc[0].astype(int).values,
    'usuarios_secuencial_sin_orden': conversion.iloc[0, :6].astype(int).values,
    'usuarios_secuencial_con_orden': funnel_seq.iloc[0].astype(int).values
})
base = tabla_funnel['usuarios_secuencial_con_orden']
tabla_funnel['conv_vs_anterior_%'] = (base / base.shift(1) * 100).round(2)
tabla_funnel['dropoff_%'] = (100 - tabla_funnel['conv_vs_anterior_%']).round(2)
tabla_funnel['conv_acumulada_%'] = (base / base.iloc[0] * 100).round(2)
tabla_funnel

,etapa,usuarios_alcance,usuarios_secuencial_sin_orden,usuarios_secuencial_con_orden,conv_vs_anterior_%,dropoff_%,conv_acumulada_%
0,first_visit,7796,7796,7796,NaN,NaN,100.00
1,select_item,7582,7393,6728,86.30,13.70,86.30
2,add_to_cart,7634,7052,4983,74.06,25.94,63.92
3,begin_checkout,7208,6364,2663,53.44,46.56,34.16
4,add_payment_info,6250,4967,919,34.51,65.49,11.79
5,purchase,6240,3857,258,28.07,71.93,3.31


# 🔹 Paso 3: Análisis de funnel de conversión


**📊 Parte 1: Construcción del funnel**

Se parte de 7.796 usuarios con `first_visit` (los otros 204 usuarios de la tabla no
registran ese evento). El conteo bruto de usuarios únicos por evento no respeta la
secuencia (p. ej., `add_to_cart` supera a `select_item` y el 80% parecería comprar),
por lo que se construyó un funnel secuencial: un usuario avanza a una etapa si tiene
ese evento en una fecha igual o posterior a la etapa anterior, tomando la primera
fecha válida. Se usa `>=` porque los datos solo incluyen fecha, sin hora.

| Etapa | Usuarios (bruto) | Usuarios (secuencial) | % sobre first_visit |
|---|---|---|---|
| first_visit | 7.796 | 7.796 | 100,00% |
| select_item | 7.582 | 6.728 | 86,30% |
| add_to_cart | 7.634 | 4.983 | 63,92% |
| begin_checkout | 7.208 | 2.663 | 34,16% |
| add_payment_info | 6.250 | 919 | 11,79% |
| purchase | 6.240 | 258 | 3,31% |

---

**📉 Parte 2: Análisis de conversión**

- **Mayor pérdida en volumen:** `add_to_cart` → `begin_checkout`, con 2.320 usuarios
  (46,56%).
- **Mayor pérdida porcentual:** `add_payment_info` → `purchase`, con 71,93%.
  `begin_checkout` → `add_payment_info` también es alta (65,49%).
- **Tasa de conversión final: 3,31%** (258 de 7.796). De los 2.663 usuarios que inician
  checkout, solo 258 (9,7%) compran, así que el checkout concentra la mayor fricción.
- **Posibles causas:** problemas en la pasarela o en los métodos de pago aceptados. Además,
  los conteos brutos de `add_payment_info` (6.250) y `purchase` (6.240) son casi iguales y
  muy superiores a los del funnel secuencial (919 y 258), lo que sugiere eventos fuera de
  secuencia o faltantes, es decir, que parte de la pérdida podría deberse a fallos de
  tracking. Se recomienda revisar tanto la pasarela como la instrumentación de los eventos
  de checkout.

**Limitaciones:** granularidad diaria (los eventos del mismo día se aceptan en cualquier
orden), ventana de tiempo abierta y eventos faltantes en compradores.

---

---

# 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users`
- `user_activity`

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [ ]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH usuarios AS (
    SELECT
        id_usuario,
        CAST(fecha_registro AS DATE) AS fecha_registro,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte
    FROM users
),
limite AS (
    SELECT MAX(CAST(fecha_actividad AS DATE)) AS ultima_fecha
    FROM user_activity
),
-- Solo usuarios con las 3 semanas ya observables (evita cohortes incompletas)
usuarios_completos AS (
    SELECT u.*
    FROM usuarios u
    CROSS JOIN limite l
    WHERE u.fecha_registro + 21 <= l.ultima_fecha
),
-- Actividad efectiva, asignada a la semana 1, 2 o 3
actividad AS (
    SELECT
        id_usuario,
        CEIL(dias_despues_registro / 7.0) AS semana
    FROM user_activity
    WHERE activo = 1
      AND dias_despues_registro BETWEEN 1 AND 21
)
SELECT
    u.cohorte,
    COUNT(DISTINCT u.id_usuario) AS usuarios_iniciales,

    COUNT(DISTINCT CASE WHEN a.semana = 1 THEN u.id_usuario END) AS retenido_w1,
    COUNT(DISTINCT CASE WHEN a.semana = 2 THEN u.id_usuario END) AS retenido_w2,
    COUNT(DISTINCT CASE WHEN a.semana = 3 THEN u.id_usuario END) AS retenido_w3,

    ROUND(100.0 * COUNT(DISTINCT CASE WHEN a.semana = 1 THEN u.id_usuario END)
          / COUNT(DISTINCT u.id_usuario), 2) AS semana_1,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN a.semana = 2 THEN u.id_usuario END)
          / COUNT(DISTINCT u.id_usuario), 2) AS semana_2,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN a.semana = 3 THEN u.id_usuario END)
          / COUNT(DISTINCT u.id_usuario), 2) AS semana_3
FROM usuarios_completos u
LEFT JOIN actividad a ON a.id_usuario = u.id_usuario
GROUP BY u.cohorte
ORDER BY u.cohorte;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,usuarios_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01 00:00:00+00:00,1627,697,668,656,42.84,41.06,40.32
1,2025-02-01 00:00:00+00:00,1444,611,609,635,42.31,42.17,43.98
2,2025-03-01 00:00:00+00:00,1636,677,705,690,41.38,43.09,42.18
3,2025-04-01 00:00:00+00:00,1606,680,697,663,42.34,43.40,41.28
4,2025-05-01 00:00:00+00:00,1687,695,676,706,41.20,40.07,41.85



# 🔹 Paso 4: Análisis retención por cohortes
- **Definición:** un usuario cuenta como retenido en la semana N si estuvo activo durante esa semana (no es acumulado). Solo se incluyen cohortes con las 3 semanas ya observables, por eso la cohorte de junio queda fuera.
- La retención se mantiene estable en **~40–44%** en las cinco cohortes de ene–may 2025, sin diferencias relevantes entre ellas y sin decaer entre la semana 1 y la 3, lo que sugiere un grupo estable de usuarios activos.
- **Siguiente paso sugerido:** segmentar por `tipo_plan` (tabla `users`) para ver si la estabilidad se mantiene entre planes.

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado**
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** La tasa de conversión del grupo control es igual a la del grupo tratamiento.
   - **H₁ (Hipótesis alternativa):** La tasa de conversión del grupo tratamiento es distinta a la del grupo control.
   
**Test estadístico:** prueba Z
**Nivel de significancia alpha:** 0.05

In [ ]:
# Carga y exploración del dataframe del experimento (dataset del curso; enlace omitido)
ex = pd.read_csv('data/experiment_checkout_ui.csv')
ex.head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [ ]:
# Revisión de calidad de los datos
print('Filas:', ex.shape[0])
print(ex.isna().sum())
print('Duplicados exactos:', ex.duplicated().sum(), '| id_usuario repetidos:', ex['id_usuario'].duplicated().sum())
print('Valores de convirtio:', sorted(ex['convirtio'].unique()))

Filas: 10000
id_usuario         0
variante           0
convirtio          0
dispositivo        0
pais               0
duracion_sesion    0
timestamp          0
dtype: int64
Duplicados exactos: 0 | id_usuario repetidos: 0
Valores de convirtio: [np.int64(0), np.int64(1)]


In [ ]:
# Revisión de registros por variables categóricas para asegurar representatividad antes de realizar la prueba.
print (ex['dispositivo'].value_counts())
print (ex['pais'].value_counts())
print(ex['variante'].value_counts())

dispositivo
desktop    5042
mobile     4958
Name: count, dtype: int64
pais
Mexico       3405
Argentina    3317
Colombia     3278
Name: count, dtype: int64
variante
tratamiento    5035
control        4965
Name: count, dtype: int64


In [ ]:
# Sample Ratio Mismatch para revisar si la asignación 50/50 es compatible con el azar
from scipy.stats import chisquare
conteo_grupos = ex['variante'].value_counts()
chi2, p_srm = chisquare(conteo_grupos.values)
print(f'SRM: chi2={chi2:.3f}, p={p_srm:.3f} ->', 'asignación desbalanceada, revisar' if p_srm < 0.01 else 'sin señales de desbalance')
# Balance de covariables entre variantes
print(pd.crosstab(ex['variante'], ex['dispositivo'], normalize='index').round(3))
print(pd.crosstab(ex['variante'], ex['pais'], normalize='index').round(3))

SRM: chi2=0.490, p=0.484 -> sin señales de desbalance
dispositivo  desktop  mobile
variante                    
control        0.505   0.495
tratamiento    0.503   0.497
pais         Argentina  Colombia  Mexico
variante                                
control          0.334     0.329   0.338
tratamiento      0.330     0.327   0.343


In [ ]:
# importar librería para prueba
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep, proportion_effectsize

alpha = 0.05
resumen = ex.groupby('variante')['convirtio'].agg(conversiones='sum', usuarios='count')
resumen['tasa'] = resumen['conversiones'] / resumen['usuarios']
print(resumen.round(4))

x_t, n_t = resumen.loc['tratamiento', ['conversiones', 'usuarios']]
x_c, n_c = resumen.loc['control', ['conversiones', 'usuarios']]
tasa_t, tasa_c = x_t / n_t, x_c / n_c

z_stat, p_value = proportions_ztest([x_t, x_c], [n_t, n_c])          # bilateral
ic_bajo, ic_alto = confint_proportions_2indep(x_t, n_t, x_c, n_c, compare='diff', method='wald')
h_observado = proportion_effectsize(tasa_t, tasa_c)                    # h de Cohen

print(f'\nEstadístico z: {z_stat:.4f} | Valor p: {p_value:.4f}')
print(f'Diferencia (trat − control): {tasa_t - tasa_c:+.4f}  |  IC 95%: [{ic_bajo:+.4f}, {ic_alto:+.4f}]')
print(f"Tamaño del efecto (h de Cohen): {h_observado:.4f}")
print('Decisión:', 'Rechazamos H0' if p_value < alpha else 'No rechazamos H0 (no hay evidencia suficiente de una diferencia)')

             conversiones  usuarios    tasa
variante                                   
control               779      4965  0.1569
tratamiento           820      5035  0.1629

Estadístico z: 0.8133 | Valor p: 0.4161
Diferencia (trat − control): +0.0060  |  IC 95%: [-0.0084, +0.0203]
Tamaño del efecto (h de Cohen): 0.0163
Decisión: No rechazamos H0 (no hay evidencia suficiente de una diferencia)


# Interpretación del resultado
- Tasa de conversión: control **15.69%**, tratamiento **16.29%**; diferencia **+0.0060** (+0.60 pp), IC 95%: **[-0.0084, +0.0203]**; z = **0.8133**, p = **0.4161**.
- Con α = 0.05, **no se rechaza H₀**: no hay evidencia estadística de que el tratamiento cambie la tasa de conversión.
- Esto **no equivale a demostrar que no hay efecto**. El IC es compatible con desde un leve empeoramiento (-0.84 pp) hasta una mejora moderada (+2.03 pp), por lo que el resultado es **inconcluso**.
- Con ~5,000 usuarios por grupo, el test es poco sensible a mejoras pequeñas, por lo que un resultado no significativo no descarta un efecto modesto.

- **Recomendación de negocio:** definir primero el efecto mínimo que justificaría el cambio. Si bastan mejoras ≥ 2 pp, los datos son compatibles como máximo con ese efecto (límite superior del IC: +2.03 pp), por lo que no justifican adoptar el cambio por sí solos.


---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo:**  
Presentar de forma clara y visual los resultados del análisis de ventas, costos y marketing.

Dashboard en **Power BI** construido con los CSV limpios del Paso 1 (`orders_clean.csv`, `catalog_clean.csv`, `marketing_clean.csv`) y una tabla de fechas propia. Consta de dos páginas:

**Página 1 – Overview Ejecutivo**
- KPIs: revenue, costo total, gasto en marketing y utilidad después de marketing.
- Ventas por mes, y ventas, unidades vendidas y gasto de marketing por país.

**Página 2 – Análisis Comercial**
- Unidades e ingresos por categoría, ingresos por canal e ingresos por mes y categoría.
- Tablas de detalle por categoría (unidades, ticket promedio, ingresos) y por canal (ingresos, gasto, % del gasto e ingresos después de marketing).
- Filtros por país y por mes.

**Mejoras previstas**
- Modelo en estrella (tablas de país, canal y fecha relacionadas con pedidos y marketing) para que el gasto de marketing responda a los filtros.
- Visual de margen por producto, que haría visible el margen negativo de Laptop-Gaming-16GB.
- KPIs de margen %, ROAS y unidades por pedido, y una página de detalle por pedido y producto.

---

## 🤖 Uso responsable de IA en este proyecto

- **Lo que hice yo:** definí la pregunta de negocio, las reglas de limpieza, la interpretación de los resultados y las conclusiones.
- **Dónde apoyó la IA:** revisión crítica del notebook y del dashboard, sugerencias de validación (llaves de unión, orden temporal del funnel, SRM, intervalos de confianza), corrección de erratas y formato.
- **Cómo lo verifiqué:** comprobé que las cifras clave (revenue, costo y marketing) coinciden entre el notebook y el dashboard.
- **Qué decidí no incluir:** un análisis de potencia estadística sugerido por la IA, porque no pude explicarlo con seguridad; prefiero presentar solo lo que puedo defender.
- **Privacidad:** retiré credenciales y enlaces del curso antes de publicar.